In [31]:
# CELL 1 — Read the raw patients CSV

from pyspark.sql.functions import current_timestamp, lit

batch_id = "patients_batch_001"
source_file = "patients.csv"

patients_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("Files/raw/patients.csv")
    .withColumn("batch_id", lit(batch_id))
    .withColumn("source_file_name", lit(source_file))
    .withColumn("ingestion_timestamp", current_timestamp())
)

patients_df.printSchema()
print(patients_df.columns)
display(patients_df.limit(10))

StatementMeta(, 6bf6614b-575b-4298-ab00-dfe9589bd2f6, 33, Finished, Available, Finished, False)

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- EMAIL: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- home_address: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- policy_number: string (nullable = true)
 |--  is_active : string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- last_updated_timestamp: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)

['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'EMAIL', 'phone_number', 'home_address', 'insurance_provider', 'policy_number', ' is_active ', 'start_date', 'end_date

SynapseWidget(Synapse.DataFrame, c3a16f51-8e73-4c12-b703-f6003f28fb46)

In [32]:
# CELL 2 — Clean and standardize column names

patients_df = patients_df.toDF(
    *[
        column.strip()
        .lower()
        .replace(" ", "_")
        for column in patients_df.columns
    ]
)

print(patients_df.columns)

StatementMeta(, 6bf6614b-575b-4298-ab00-dfe9589bd2f6, 34, Finished, Available, Finished, False)

['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'email', 'phone_number', 'home_address', 'insurance_provider', 'policy_number', 'is_active', 'start_date', 'end_date', 'last_updated_timestamp', 'unexpected_source_field', 'batch_id', 'source_file_name', 'ingestion_timestamp']


In [33]:
# CELL 3 — Write the cleaned DataFrame to the Bronze Delta table

spark.sql("DROP TABLE IF EXISTS bronze_patients")

(
    patients_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_patients")
)

print("Bronze rows:", spark.table("bronze_patients").count())
print("bronze_patients created successfully.")

StatementMeta(, 6bf6614b-575b-4298-ab00-dfe9589bd2f6, 35, Finished, Available, Finished, False)

Bronze rows: 540743
bronze_patients created successfully.


In [34]:
# CELL 4 — Verify the Bronze table

bronze_patients_df = spark.table("bronze_patients")

bronze_patients_df.printSchema()
print(bronze_patients_df.columns)
display(bronze_patients_df.limit(10))

StatementMeta(, 6bf6614b-575b-4298-ab00-dfe9589bd2f6, 36, Finished, Available, Finished, False)

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- home_address: string (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- policy_number: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- last_updated_timestamp: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'email', 'phone_number', 'home_address', 'insurance_provider', 'policy_number', 'is_active', 'start_date', 'end_date', 'las

SynapseWidget(Synapse.DataFrame, 8371cc48-f587-48b6-b783-9c752b30e700)

In [35]:
# CELL 5 — Validate Bronze ingestion

source_row_count = patients_df.count()
bronze_row_count = bronze_patients_df.count()

print("Source rows:", source_row_count)
print("Bronze rows:", bronze_row_count)

if source_row_count == bronze_row_count:
    print("Validation passed: all source rows were written to Bronze.")
else:
    print("Validation failed: row counts do not match.")

StatementMeta(, 6bf6614b-575b-4298-ab00-dfe9589bd2f6, 37, Finished, Available, Finished, False)

Source rows: 540743
Bronze rows: 540743
Validation passed: all source rows were written to Bronze.
